<a href="https://colab.research.google.com/github/rmvjh27/protein-analysis-notebooks/blob/main/PDB_Sequence_Parser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Protein Sequence Extractor from PDB files
Extracts the amino acid sequence from a protein structure PDB file. The extracted sequence can be used for alignment purposes (MSA, sequence alignment for MODELLER, etc.)

## Define function for sequence extraction

In [ ]:
def extract_amino_acid_sequences_from_pdb_content(pdb_content):
    """
    Extracts amino acid sequences from the ATOM records of PDB content,
    handling multiple chains and ensuring correct residue order.

    Args:
        pdb_content (str): The content of a PDB file as a string.

    Returns:
        dict: A dictionary where keys are chain IDs (str) and values are
              the amino acid sequences (str) for that chain, with one-letter
              codes.
    """
    # Mapping for three-letter to one-letter amino acid codes
    three_to_one_letter_map = {
        'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
        'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
        'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
        'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
    }

    chain_residues = {}
    lines = pdb_content.split('\n')

    for line in lines:
        if line.startswith('ATOM '):
            # According to PDB format:
            # Residue name: columns 18-20 (0-indexed 17-19)
            # Chain ID: column 22 (0-indexed 21)
            # Residue sequence number: columns 23-26 (0-indexed 22-25)
            residue_name_3letter = line[17:20].strip()
            chain_id = line[21].strip()
            residue_seq_num = int(line[22:26].strip())

            if chain_id not in chain_residues:
                chain_residues[chain_id] = set() # Use a set to store unique (seq_num, name) tuples

            # Convert to one-letter code if available, otherwise keep three-letter or a placeholder
            residue_name_1letter = three_to_one_letter_map.get(residue_name_3letter, 'X')
            chain_residues[chain_id].add((residue_seq_num, residue_name_1letter))

    # Process the extracted residues to form sequences
    amino_acid_sequences = {}
    for chain_id, residues_set in chain_residues.items():
        # Convert set to list and sort by residue sequence number
        sorted_residues = sorted(list(residues_set), key=lambda x: x[0])
        # Extract only the residue names and join them to form the sequence
        # No hyphens for one-letter FASTA format
        sequence = ''.join([res[1] for res in sorted_residues])
        amino_acid_sequences[chain_id] = sequence

    return amino_acid_sequences

## Upload your structure file here

In [ ]:
# import the files functionality
from google.colab import files

# code for uploading PDB file
pdb_files_dict = files.upload()

# Get the filename from the dictionary keys
if pdb_files_dict:
    pdb_filename = list(pdb_files_dict.keys())[0]
else:
    print("No file uploaded.")
    pdb_filename = None

# Read the PDB file if a filename was obtained
pdb_file_content = None
if pdb_filename:
    with open(pdb_filename, 'r') as f:
        pdb_file_content = f.read()

if pdb_file_content:
    amino_acids_from_file = extract_amino_acid_sequences_from_pdb_content(pdb_file_content)

    print("Amino Acid Sequences (FASTA format - one-letter code):\n")

    if amino_acids_from_file:
        for chain, seq in amino_acids_from_file.items():
            print(f">Chain_{chain}")
            # Split the sequence into chunks for FASTA formatting
            # Sequence is already in one-letter format without hyphens
            for i in range(0, len(seq), 60):
                print(seq[i:i+60])
    else:
        print("No amino acid sequences found.")
